In [1]:
!pip install -q kaggle

In [2]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("KGAT_1c9f6c4bbf373df37be2179372fb2751")

KGAT_1c9f6c4bbf373df37be2179372fb2751··········


In [3]:
!kaggle datasets download -d grassknoted/asl-alphabet -p /content/asl --unzip

Dataset URL: https://www.kaggle.com/datasets/grassknoted/asl-alphabet
License(s): GPL-2.0
100% 1.03G/1.03G [00:26<00:00, 41.9MB/s]



Libraries and device

In [4]:
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split
import torch

raw = Path("/content/asl/asl_alphabet_train/asl_alphabet_train")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = datasets.ImageFolder(raw, transform=transform)

# Select 200 images per class
indices = []
for class_index in range(len(dataset.classes)):
    class_indices = [
        i for i, (_, label) in enumerate(dataset.samples)
        if label == class_index
    ]
    indices.extend(class_indices[:200])

small_dataset = Subset(dataset, indices)

train_size = int(0.8 * len(small_dataset))
val_size = len(small_dataset) - train_size

train_dataset, val_dataset = random_split(
    small_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

num_classes = len(dataset.classes)

print("Number of classes:", num_classes)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

Number of classes: 29
Training images: 4640
Validation images: 1160


Load the pretrained ResNet18 Model

In [5]:
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze pretrained layers
for param in model.parameters():
    param.requires_grad = False

# Replace the final layer
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)

print("Device:", device)
print("ResNet18 ready!")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 182MB/s]


Device: cuda
ResNet18 ready!


Loss function and optimizer

In [6]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

Model Train

In [7]:
import time

epochs = 3

for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0
    start = time.time()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {running_loss/total:.4f} "
        f"Accuracy: {100*correct/total:.2f}% "
        f"Time: {time.time()-start:.2f}s"
    )

Epoch [1/3] Loss: 1.5619 Accuracy: 68.94% Time: 22.36s
Epoch [2/3] Loss: 0.4368 Accuracy: 96.10% Time: 19.16s
Epoch [3/3] Loss: 0.2411 Accuracy: 97.93% Time: 18.73s


Model Evaluation

In [8]:
model.eval()

correct = 0
total = 0
val_loss = 0.0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

print("Validation Loss:", round(val_loss / total, 4))
print("Validation Accuracy:", round(100 * correct / total, 2), "%")

Validation Loss: 0.1413
Validation Accuracy: 98.53 %


Saving the model

In [9]:
from pathlib import Path

Path("/content/results").mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "classes": dataset.classes
}, "/content/results/resnet18_asl.pth")

print("ResNet18 model saved!")

ResNet18 model saved!


In [10]:
from google.colab import files

files.download("/content/results/resnet18_asl.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>